# Web-Gold-40K recovery v2.8 reviewed 5K mini

This is Step 7 only. It requires either a passed two-person correction overlay or an explicit two-person no-change review attestation, plus the accepted RETRY/ABORT supplement v2. It trains 5,000 rows for five zero-based epochs (0–4), selects checkpoints only on 500 reviewed original-Gold validation rows, evaluates all 194 supplement validation rows separately, and reads zero locked-test rows. A failed quality gate saves all evidence but blocks full training.

In [ ]:
# 1. Reject the wrong accelerator before cloning, installing, or loading data.
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'T4' in GPU_NAME.upper(), f'Tesla T4 required; stop this session (found {GPU_NAME}).'

In [ ]:
# 2. Pull the reviewed Code branch and record the physical environment.
from pathlib import Path
import importlib
import importlib.metadata as metadata
import json, os, subprocess, sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(REPO_ROOT)], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.environ['PYTHONPATH'] = str(SOURCE_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
importlib.invalidate_caches()
os.chdir(REPO_ROOT)
import web_agent
assert Path(web_agent.__file__).resolve().is_relative_to(SOURCE_ROOT.resolve())
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
environment = {
    'git_commit': COMMIT,
    'gpu': GPU_NAME,
    'python': sys.version.split()[0],
    **{name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']},
}
ENVIRONMENT_PATH = Path('/kaggle/working/gold_recovery_v2_8_environment.json')
ENVIRONMENT_PATH.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))

In [ ]:
# 3. Resolve data and register either a passed overlay or an honest no-change review attestation.
from web_agent.data.recovery_supplement import resolve_supplement_root
from web_agent.data.review_overlay import ReviewOverlay
from datetime import datetime, timezone

INPUT_ROOT = Path('/kaggle/input')
ORIGINAL_ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SUPPLEMENT_ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/gold-40k-retry')
DEVELOPMENT_SPLITS = ('split_train.json', 'split_val.json')
# Keep these values only when two people completed the original-data review
# and approved it without requesting any correction, exclusion, or recollection.
NO_CHANGE_REVIEW_CONFIRMED = True
NO_CHANGE_REVIEWER_COUNT = 2

def find_original_root(root: Path) -> Path:
    candidates = []
    if root.is_dir() and all((root / name).is_file() for name in DEVELOPMENT_SPLITS):
        candidates.append(root.resolve())
    if root.is_dir():
        for current, _, files in os.walk(root, followlinks=True):
            if set(DEVELOPMENT_SPLITS).issubset(files):
                candidates.append(Path(current).resolve())
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f'Expected one original train/val root; found {candidates}'
    return candidates[0]

ORIGINAL_ROOT = find_original_root(ORIGINAL_ATTACHED_ROOT)
SUPPLEMENT_ROOT = resolve_supplement_root(SUPPLEMENT_ATTACHED_ROOT)
reconciliation_reports = sorted(INPUT_ROOT.rglob('review_reconciliation_report.json'))
assert len(reconciliation_reports) <= 1, f'Attach at most one reconciliation output; found {reconciliation_reports}'
if reconciliation_reports:
    RECONCILIATION_DIR = reconciliation_reports[0].parent.resolve()
    overlay = ReviewOverlay.load(RECONCILIATION_DIR)
    assert overlay.report['status'] == 'PASS'
    assert overlay.report['controlled_mini_permitted'] is True
    assert overlay.report['test_rows_read'] == 0
    REVIEW_MODE = 'artifact_verified_overlay'
    review_attestation = {
        'status': 'PASS',
        'review_mode': REVIEW_MODE,
        'overlay_provenance': overlay.provenance(),
        'test_rows_read': 0,
    }
else:
    assert NO_CHANGE_REVIEW_CONFIRMED is True, (
        'No reconciliation package is attached. Confirm the completed '
        'two-person no-change review before continuing.'
    )
    assert NO_CHANGE_REVIEWER_COUNT == 2, 'Exactly two reviewers are required.'
    RECONCILIATION_DIR = None
    overlay = None
    REVIEW_MODE = 'two_person_manual_no_change_attestation'
    review_attestation = {
        'status': 'PASS',
        'review_mode': REVIEW_MODE,
        'reviewers': NO_CHANGE_REVIEWER_COUNT,
        'owner_confirmed_no_corrections': True,
        'owner_confirmed_no_exclusions': True,
        'source_records_mutated': False,
        'test_rows_read': 0,
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
        'limitation': (
            'Manual no-change confirmation has no row-level reconciliation '
            'ledger; report it as an attestation, not an artifact-verified overlay.'
        ),
    }
REVIEW_ATTESTATION_PATH = Path('/kaggle/working/gold_recovery_v2_8_review_attestation.json')
REVIEW_ATTESTATION_PATH.write_text(json.dumps(review_attestation, indent=2), encoding='utf-8')
print('ORIGINAL_ROOT =', ORIGINAL_ROOT)
print('SUPPLEMENT_ROOT =', SUPPLEMENT_ROOT)
print('REVIEW_MODE =', REVIEW_MODE)
print('RECONCILIATION_DIR =', RECONCILIATION_DIR or 'not required: review approved with no changes')
print(json.dumps(review_attestation, indent=2))

In [ ]:
# 4. Revalidate both data gates in this exact Kaggle session. Test remains unread.
OVERLAY_VALIDATION_PATH = Path('/kaggle/working/gold_v2_8_overlay_validation.json')
SUPPLEMENT_VALIDATION_PATH = Path('/kaggle/working/gold_v2_8_supplement_validation.json')
MULTISOURCE_AUDIT_PATH = Path('/kaggle/working/gold_v2_8_multisource_audit.json')
commands = [
    [
        sys.executable, 'scripts/validate_retry_abort_supplement.py',
        '--supplement-root', str(SUPPLEMENT_ROOT),
        '--report', str(SUPPLEMENT_VALIDATION_PATH),
    ],
    [
        sys.executable, 'scripts/audit_retry_abort_multisource.py',
        '--original-root', str(ORIGINAL_ROOT),
        '--supplement-root', str(SUPPLEMENT_ROOT),
        '--report', str(MULTISOURCE_AUDIT_PATH),
    ],
]
gate_paths = [SUPPLEMENT_VALIDATION_PATH, MULTISOURCE_AUDIT_PATH]
if RECONCILIATION_DIR is not None:
    commands.insert(0, [
        sys.executable, 'scripts/validate_gold_review_overlay.py',
        '--data-root', str(ORIGINAL_ROOT),
        '--reconciliation-dir', str(RECONCILIATION_DIR),
        '--report', str(OVERLAY_VALIDATION_PATH),
    ])
    gate_paths.insert(0, OVERLAY_VALIDATION_PATH)
for command in commands:
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
for report_path in gate_paths:
    gate = json.loads(report_path.read_text(encoding='utf-8'))
    assert gate['status'] == 'PASS', f'Gate failed: {report_path}'
    assert gate['test_rows_read'] == 0
print('REVIEW + SUPPLEMENT + MULTISOURCE GATES PASSED')

In [ ]:
# 5. Preregister the immutable Step-7 run contract.
from web_agent.config import load_config
from web_agent.data.gold_dataloader import load_gold_split, load_gold_split_sources

CONFIG_PATH = 'configs/backbones/qwen2vl_2b_gold_v2_8.yaml'
TRAIN_ROWS = 5_000
ORIGINAL_VAL_ROWS = 500
SUPPLEMENT_VAL_ROWS = 194
EPOCHS = 5
SEED = 42
MIN_PIXELS = 50_176
MAX_PIXELS = 200_704  # same 448px mini profile used by the v2.7 controlled run
cfg = load_config(CONFIG_PATH)
cfg['data']['root'] = str(ORIGINAL_ROOT)
if RECONCILIATION_DIR is not None:
    cfg['data']['review_overlay_dir'] = str(RECONCILIATION_DIR)
else:
    cfg['data'].pop('review_overlay_dir', None)
cfg['data']['recovery_supplement']['root'] = str(SUPPLEMENT_ROOT)
cfg['data']['recovery_supplement']['include_in_primary_validation'] = False
cfg['data']['num_workers'] = 4
assert cfg['train']['controlled_experiment_tag'] == 'recovery_v2_8'
assert cfg['train']['quality_selection_rule'] == 'all_gates_then_outcome_mcc'
assert cfg['data']['recovery_transitions'] is True
assert cfg['loss']['hierarchical_recovery'] is True
assert cfg['loss']['recovery_outcome'] == 0.09
sources = load_gold_split_sources(cfg, 'val')
reviewed_train_rows = len(load_gold_split(cfg, 'train'))
reviewed_val_rows = len(load_gold_split(cfg, 'val'))
assert 5_000 <= reviewed_train_rows <= 24_107  # reviewed exclusions may reduce original rows
assert 500 <= reviewed_val_rows <= 7_861
assert len(sources['retry_abort_supplement_v2']) == SUPPLEMENT_VAL_ROWS
contract = {
    'config': CONFIG_PATH,
    'git_commit': COMMIT,
    'seed': SEED,
    'train_rows': TRAIN_ROWS,
    'reviewed_combined_train_rows_available': reviewed_train_rows,
    'original_validation_rows': ORIGINAL_VAL_ROWS,
    'reviewed_original_validation_rows_available': reviewed_val_rows,
    'supplement_validation_rows': SUPPLEMENT_VAL_ROWS,
    'epochs_zero_based': [0, 1, 2, 3, 4],
    'min_pixels': MIN_PIXELS,
    'max_pixels': MAX_PIXELS,
    'checkpoint_selection_source': 'reviewed original Gold validation only',
    'review_mode': REVIEW_MODE,
    'review_attestation': str(REVIEW_ATTESTATION_PATH),
    'test_rows_read': 0,
    'full_training_started': False,
}
CONTRACT_PATH = Path('/kaggle/working/gold_recovery_v2_8_contract.json')
CONTRACT_PATH.write_text(json.dumps(contract, indent=2), encoding='utf-8')
print(json.dumps(contract, indent=2))

# Start the mini in this same cell so Kaggle version runs cannot stop between preflight and training.
RESULT_CSV = Path('/kaggle/working/gold_recovery_v2_8_metrics.csv')
DIAGNOSTICS_JSON = Path('/kaggle/working/gold_recovery_v2_8_diagnostics.json')
FULL_REPORT_JSON = Path('/kaggle/working/gold_recovery_v2_8_report.json')
SOURCE_VALIDATION_CSV = Path('/kaggle/working/gold_recovery_v2_8_source_validation.csv')
command = [
    sys.executable, 'scripts/run_gold.py',
    '--config', CONFIG_PATH,
    '--stage', 'mini',
    '--data-root', str(ORIGINAL_ROOT),
    '--supplement-root', str(SUPPLEMENT_ROOT),
    '--train-rows', str(TRAIN_ROWS),
    '--val-rows', str(ORIGINAL_VAL_ROWS),
    '--epochs', str(EPOCHS),
    '--min-pixels', str(MIN_PIXELS),
    '--max-pixels', str(MAX_PIXELS),
    '--result-csv', str(RESULT_CSV),
    '--diagnostics-json', str(DIAGNOSTICS_JSON),
    '--report-json', str(FULL_REPORT_JSON),
    '--source-validation-csv', str(SOURCE_VALIDATION_CSV),
]
if RECONCILIATION_DIR is not None:
    command.extend(['--review-overlay-dir', str(RECONCILIATION_DIR)])
print('Running:', ' '.join(command))
result = subprocess.run(command, check=False)
assert result.returncode == 0, f'Mini runner crashed with exit code {result.returncode}'
assert FULL_REPORT_JSON.is_file(), 'Mini completed without a full report'

In [ ]:
# 6. The mini was launched automatically at the end of Cell 5.
# This cell is intentionally a no-op checkpoint for Kaggle version runs.
assert FULL_REPORT_JSON.is_file(), 'Cell 5 must finish the mini before results are checked.'
print('Mini report ready:', FULL_REPORT_JSON)

In [ ]:
# 7. Verify artifacts and make the full-training decision from registered gates.
report = json.loads(FULL_REPORT_JSON.read_text(encoding='utf-8'))
assert report['test_rows_read'] == 0
assert report['train_rows'] == TRAIN_ROWS
assert report['val_rows'] == ORIGINAL_VAL_ROWS
assert len(report['history']) == EPOCHS
assert [int(row['epoch']) for row in report['history']] == [0, 1, 2, 3, 4]
assert report['train_distribution']['source_dataset']['retry_abort_supplement_v2'] == 608
assert report['source_validation']['primary_original_gold']['rows'] == ORIGINAL_VAL_ROWS
assert report['source_validation']['primary_original_gold']['checkpoint_selection_source'] is True
supplement_validation = report['source_validation']['supplement_retry_abort']
assert supplement_validation['rows'] == SUPPLEMENT_VAL_ROWS
assert supplement_validation['test_rows_read'] == 0
assert report['review_overlay']['enabled'] is (RECONCILIATION_DIR is not None)
assert all(path.is_file() for path in (RESULT_CSV, DIAGNOSTICS_JSON, SOURCE_VALIDATION_CSV))
report['manual_review_attestation'] = review_attestation
FULL_REPORT_JSON.write_text(json.dumps(report, indent=2), encoding='utf-8')

display_metrics = (
    'outcome_mcc', 'failure_macro_f1', 'action_macro_f1',
    'strategy_attempted_macro_f1', 'recovery_outcome_mcc',
    'memory_mcc', 'bbox_mean_iou', 'bbox_recall_iou50', 'outcome_ece',
)
for epoch in report['history']:
    values = '  '.join(f'{name}={float(epoch[name]):.4f}' for name in display_metrics)
    print(f"epoch {int(epoch['epoch'])}: {values}")
print('eligible epochs:', report['quality_gates']['eligible_epochs'])
print('selected epoch:', report['selected_epoch'])
print('selected checkpoint:', report['best_checkpoint'])
print('supplement validation:', json.dumps(supplement_validation['metrics'], indent=2))

failed_checks = [name for name, passed in report['quality_gates']['checks'].items() if not passed]
decision = {
    'status': report['status'],
    'training_disposition': report['training_disposition'],
    'review_mode': REVIEW_MODE,
    'review_limitation': review_attestation.get('limitation'),
    'eligible_epochs': report['quality_gates']['eligible_epochs'],
    'selected_epoch': report['selected_epoch'],
    'selected_checkpoint': report['best_checkpoint'],
    'failed_selected_epoch_checks': failed_checks,
    'test_rows_read': 0,
    'outputs': [str(RESULT_CSV), str(DIAGNOSTICS_JSON), str(FULL_REPORT_JSON), str(SOURCE_VALIDATION_CSV), str(REVIEW_ATTESTATION_PATH)],
}
DECISION_PATH = Path('/kaggle/working/gold_recovery_v2_8_decision.json')
DECISION_PATH.write_text(json.dumps(decision, indent=2), encoding='utf-8')
print(json.dumps(decision, indent=2))
assert report['status'] == 'PASS', (
    'V2.8 MINI FINISHED BUT DID NOT PASS ALL GATES. '
    f'Full training remains blocked. Failed checks: {failed_checks}. '
    'Download the saved outputs for diagnosis.'
)
print('V2.8 REVIEWED MINI PASSED. Full training may now be planned; it has not started.')